In [ ]:

# ARLINGTON / DFW ENERGY PIPELINE + DASHBOARD


# !pip install geopandas folium requests shapely pyproj fiona pandas numpy plotly branca

import os
import json
import warnings
import requests
import numpy as np
import pandas as pd
import geopandas as gpd
import folium
import plotly.express as px
import plotly.figure_factory as ff

from folium.plugins import MarkerCluster, MiniMap, Fullscreen
from IPython.display import display, HTML, Markdown

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

In [ ]:

BUILDINGS_CSV = "/content/arlington_buildings_10000+.csv"   # change if needed
OUTPUT_DIR = "/content/arlington_energy_outputs"

CENSUS_API_KEY = ""          # optional
NREL_API_KEY = "DEMO_KEY"    # replace with your own if possible

STATE_FIPS = "48"  # Texas

TARGET_COUNTIES = {
    "439": "Tarrant",
    "113": "Dallas",
    "121": "Denton",
    "085": "Collin",
    "397": "Rockwall",
    "251": "Johnson",
    "367": "Parker",
    "139": "Ellis"
}

CRS_WGS84 = "EPSG:4326"
CRS_TEXAS_FT = "EPSG:2276"   # Texas North Central (feet)

ONE_MILE_FT = 5280
THREE_MILE_FT = 15840

ROOF_USABLE_PCT = 0.80
CANOPY_COVERAGE_PCT = 0.60
PANEL_EFFICIENCY = 0.20
PERFORMANCE_RATIO = 0.80
ANNUAL_SOLAR_RADIATION = 1700
AVG_HOUSEHOLD_KWH_PER_YEAR = 12000

SITE_COVERAGE_BY_CATEGORY = {
    "big_box": 0.22,
    "strip_mall": 0.18,
    "public_asset": 0.20,
    "other_commercial": 0.25,
    "unknown": 0.25
}

os.makedirs(OUTPUT_DIR, exist_ok=True)


In [ ]:

# 2.  FUNCTIONS


def safe_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip()

def first_nonempty(row, candidates):
    for c in candidates:
        if c in row.index:
            v = safe_text(row[c])
            if v != "":
                return v
    return ""

def infer_store_name(row):
    for c in ["name", "brand", "official_name", "alt_name", "short_name"]:
        if c in row.index:
            v = safe_text(row[c])
            if v:
                return v

    # better fallback name
    if "Building_Sq_Ft" in row.index:
        return f"Asset_{row.name}_({int(row['Building_Sq_Ft']):,} sqft)"
    return f"Asset_{row.name}"

def build_address(row):
    full_addr = first_nonempty(row, ["addr:full"])
    if full_addr:
        return full_addr

    house = first_nonempty(row, ["addr:housenumber"])
    street = first_nonempty(row, ["addr:street"])
    city = first_nonempty(row, ["addr:city"])
    state = first_nonempty(row, ["addr:state"])
    zipc = first_nonempty(row, ["addr:postcode"])

    parts = []
    line1 = " ".join([p for p in [house, street] if p]).strip()
    if line1:
        parts.append(line1)
    if city:
        parts.append(city)
    if state:
        parts.append(state)
    if zipc:
        parts.append(zipc)

    return ", ".join(parts)

def classify_asset(row):
    """
    Improved classification:
    uses name + tags + building size
    """
    name = first_nonempty(row, ["Store_Name", "name", "brand", "official_name"]).lower()
    shop = first_nonempty(row, ["shop"]).lower()
    amenity = first_nonempty(row, ["amenity"]).lower()
    building = first_nonempty(row, ["building", "building:use"]).lower()
    sqft = row.get("Building_Sq_Ft", np.nan)

    text = " ".join([name, shop, amenity, building])

    big_box_keywords = [
        "walmart", "target", "costco", "sam's club", "home depot",
        "lowe", "lowes", "best buy", "kroger", "ikea", "academy",
        "aldi", "whole foods", "tom thumb", "heb", "supercenter",
        "warehouse club"
    ]

    public_keywords = [
        "school", "high school", "library", "community center",
        "municipal", "recreation center", "city hall", "college",
        "public works", "government"
    ]

    strip_keywords = [
        "shopping center", "plaza", "marketplace", "strip mall",
        "town center", "retail center", "mall"
    ]

    if any(k in text for k in big_box_keywords):
        return "big_box"

    if any(k in text for k in public_keywords):
        return "public_asset"

    if any(k in text for k in strip_keywords):
        return "strip_mall"

    if amenity in ["school", "library", "community_centre", "community_center"]:
        return "public_asset"

    if shop in ["mall", "department_store", "wholesale", "supermarket"]:
        return "big_box" if pd.notna(sqft) and sqft >= 80000 else "other_commercial"

    if building in ["retail", "commercial", "supermarket", "warehouse"]:
        if pd.notna(sqft):
            if sqft >= 100000:
                return "big_box"
            elif sqft >= 40000:
                return "other_commercial"

    # size-based fallback
    if pd.notna(sqft):
        if sqft >= 120000:
            return "big_box"
        elif sqft >= 50000:
            return "other_commercial"
        elif sqft >= 20000:
            return "strip_mall"

    return "unknown"

def sq_ft_to_sq_m(sq_ft):
    return sq_ft * 0.09290304

def solar_kwh(area_sq_ft):
    area_sq_m = sq_ft_to_sq_m(area_sq_ft)
    return area_sq_m * PANEL_EFFICIENCY * ANNUAL_SOLAR_RADIATION * PERFORMANCE_RATIO

def estimate_lot_sq_ft(row):
    cat = row["Asset_Category"]
    coverage = SITE_COVERAGE_BY_CATEGORY.get(cat, SITE_COVERAGE_BY_CATEGORY["unknown"])
    return row["Building_Sq_Ft"] / coverage

def minmax(series):
    s = series.astype(float)
    if len(s) == 0 or s.max() == s.min():
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - s.min()) / (s.max() - s.min())

def get_bbox(gdf_wgs84, margin_deg=0.08):
    minx, miny, maxx, maxy = gdf_wgs84.total_bounds
    return (minx - margin_deg, miny - margin_deg, maxx + margin_deg, maxy + margin_deg)

def human_readable_kwh(v):
    if v >= 1_000_000_000:
        return f"{v/1_000_000_000:.2f}B"
    if v >= 1_000_000:
        return f"{v/1_000_000:.2f}M"
    if v >= 1_000:
        return f"{v/1_000:.2f}K"
    return f"{v:.0f}"

In [ ]:

# 3. LOAD BUILDING DATA


df = pd.read_csv(BUILDINGS_CSV)

required_cols = ["latitude", "longitude", "Building_Sq_Ft"]
missing_req = [c for c in required_cols if c not in df.columns]
if missing_req:
    raise ValueError(f"Missing required columns: {missing_req}")

df["latitude"] = pd.to_numeric(df["latitude"], errors="coerce")
df["longitude"] = pd.to_numeric(df["longitude"], errors="coerce")
df["Building_Sq_Ft"] = pd.to_numeric(df["Building_Sq_Ft"], errors="coerce")

df = df.dropna(subset=["latitude", "longitude", "Building_Sq_Ft"]).copy()
df = df[df["Building_Sq_Ft"] > 0].copy()

df["Store_Name"] = df.apply(infer_store_name, axis=1)
df["Location"] = df.apply(build_address, axis=1)
df["Asset_Category"] = df.apply(classify_asset, axis=1)

df["Roof_Sq_Ft"] = df["Building_Sq_Ft"] * ROOF_USABLE_PCT
df["Lot_Sq_Ft"] = df.apply(estimate_lot_sq_ft, axis=1)
df["Canopy_Usable_Sq_Ft"] = (df["Lot_Sq_Ft"] - df["Building_Sq_Ft"]).clip(lower=0) * CANOPY_COVERAGE_PCT

gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["longitude"], df["latitude"]),
    crs=CRS_WGS84
)

gdf_ft = gdf.to_crs(CRS_TEXAS_FT).reset_index(drop=True)
gdf_ft["asset_id"] = np.arange(1, len(gdf_ft) + 1)

gdf_ft["buffer_1mi"] = gdf_ft.geometry.buffer(ONE_MILE_FT)
gdf_ft["buffer_3mi"] = gdf_ft.geometry.buffer(THREE_MILE_FT)

gdf_ft["buffer_1mi_area_sqft"] = gdf_ft["buffer_1mi"].area
gdf_ft["buffer_1mi_area_sqmi"] = gdf_ft["buffer_1mi_area_sqft"] / 27878400.0

gdf_ft["buffer_3mi_area_sqft"] = gdf_ft["buffer_3mi"].area
gdf_ft["buffer_3mi_area_sqmi"] = gdf_ft["buffer_3mi_area_sqft"] / 27878400.0

print(f"Loaded assets: {len(gdf_ft):,}")


Loaded assets: 357


In [ ]:

# 4. CENSUS HOUSEHOLDS


def census_get_households_for_county(state_fips, county_fips, api_key=""):
    base = "https://api.census.gov/data/2022/acs/acs5"
    params = {
        "get": "NAME,B11001_001E",
        "for": "block group:*",
        "in": f"state:{state_fips} county:{county_fips} tract:*"
    }
    if api_key:
        params["key"] = api_key

    r = requests.get(base, params=params, timeout=120)
    r.raise_for_status()
    data = r.json()

    cols = data[0]
    rows = data[1:]
    out = pd.DataFrame(rows, columns=cols)

    out["B11001_001E"] = pd.to_numeric(out["B11001_001E"], errors="coerce").fillna(0)
    out["state"] = out["state"].astype(str).str.zfill(2)
    out["county"] = out["county"].astype(str).str.zfill(3)
    out["tract"] = out["tract"].astype(str).str.zfill(6)
    out["block group"] = out["block group"].astype(str)
    out["GEOID"] = out["state"] + out["county"] + out["tract"] + out["block group"]
    out = out.rename(columns={"B11001_001E": "households"})
    return out[["GEOID", "NAME", "households"]]

def load_tiger_block_groups_for_county(state_fips, county_fips):
    tiger_zip = f"https://www2.census.gov/geo/tiger/TIGER2022/BG/tl_2022_{state_fips}_bg.zip"
    g = gpd.read_file(tiger_zip)
    g["STATEFP"] = g["STATEFP"].astype(str).str.zfill(2)
    g["COUNTYFP"] = g["COUNTYFP"].astype(str).str.zfill(3)
    g["GEOID"] = g["GEOID"].astype(str)
    g = g[(g["STATEFP"] == state_fips) & (g["COUNTYFP"] == county_fips)].copy()
    return g[["GEOID", "geometry"]]

all_bg_parts = []

for county_fips, county_name in TARGET_COUNTIES.items():
    try:
        print(f"Downloading Census + TIGER for {county_name} County...")
        hh = census_get_households_for_county(STATE_FIPS, county_fips, CENSUS_API_KEY)
        bg = load_tiger_block_groups_for_county(STATE_FIPS, county_fips)
        merged = bg.merge(hh, on="GEOID", how="left")
        merged["households"] = merged["households"].fillna(0)
        all_bg_parts.append(merged)
    except Exception as e:
        print(f"Skipped {county_name}: {e}")

if len(all_bg_parts) == 0:
    raise RuntimeError("No Census block group data could be downloaded.")

bg_gdf = pd.concat(all_bg_parts, ignore_index=True)
bg_gdf = gpd.GeoDataFrame(bg_gdf, geometry="geometry", crs=CRS_WGS84)
bg_gdf_ft = bg_gdf.to_crs(CRS_TEXAS_FT)

print(f"Loaded block groups: {len(bg_gdf):,}")


Loaded block groups: 4,212


In [ ]:

# 5. HOUSEHOLDS IN BUFFERS


def allocate_households_by_area(asset_buffers_gdf, buffer_col, block_groups_gdf, asset_id_col="asset_id"):
    buffers = asset_buffers_gdf[[asset_id_col, buffer_col]].copy()
    buffers = buffers.rename(columns={buffer_col: "geometry"})
    buffers = gpd.GeoDataFrame(buffers, geometry="geometry", crs=block_groups_gdf.crs)

    bgs = block_groups_gdf[["GEOID", "households", "geometry"]].copy()
    bgs["bg_area"] = bgs.geometry.area

    inter = gpd.overlay(buffers, bgs, how="intersection")
    inter["overlap_area"] = inter.geometry.area
    inter["allocated_households"] = np.where(
        inter["bg_area"] > 0,
        inter["households"] * (inter["overlap_area"] / inter["bg_area"]),
        0
    )

    result = inter.groupby(asset_id_col, as_index=False)["allocated_households"].sum()
    return result

gdf_ft = gdf_ft.drop(columns=[c for c in ["households_1mi", "households_3mi"] if c in gdf_ft.columns], errors="ignore")

hh_1 = allocate_households_by_area(gdf_ft, "buffer_1mi", bg_gdf_ft, "asset_id")
hh_3 = allocate_households_by_area(gdf_ft, "buffer_3mi", bg_gdf_ft, "asset_id")

hh_1 = hh_1.rename(columns={"allocated_households": "households_1mi"})
hh_3 = hh_3.rename(columns={"allocated_households": "households_3mi"})

gdf_ft = gdf_ft.merge(hh_1, on="asset_id", how="left")
gdf_ft = gdf_ft.merge(hh_3, on="asset_id", how="left")

gdf_ft["households_1mi"] = gdf_ft["households_1mi"].fillna(0).round().astype(int)
gdf_ft["households_3mi"] = gdf_ft["households_3mi"].fillna(0).round().astype(int)

print("Household calculation done.")

Household calculation done.


In [ ]:

# 6. EV STATIONS


def download_ev_stations_nrel(api_key="DEMO_KEY", state="TX"):
    url = "https://developer.nrel.gov/api/alt-fuel-stations/v1.json"
    params = {
        "api_key": api_key,
        "fuel_type": "ELEC",
        "status": "E",
        "access": "public",
        "state": f"US-{state}",
        "limit": "all"
    }
    r = requests.get(url, params=params, timeout=120)
    r.raise_for_status()
    js = r.json()

    stations = js.get("fuel_stations", [])
    ev = pd.DataFrame(stations)

    if len(ev) == 0:
        return gpd.GeoDataFrame(columns=["station_name", "geometry"], geometry="geometry", crs=CRS_WGS84)

    ev["latitude"] = pd.to_numeric(ev["latitude"], errors="coerce")
    ev["longitude"] = pd.to_numeric(ev["longitude"], errors="coerce")
    ev = ev.dropna(subset=["latitude", "longitude"]).copy()

    keep_cols = [c for c in [
        "id", "station_name", "street_address", "city", "state", "zip",
        "ev_network", "ev_level2_evse_num", "ev_dc_fast_num",
        "latitude", "longitude"
    ] if c in ev.columns]

    ev = ev[keep_cols].copy()

    ev_gdf = gpd.GeoDataFrame(
        ev,
        geometry=gpd.points_from_xy(ev["longitude"], ev["latitude"]),
        crs=CRS_WGS84
    )
    return ev_gdf

try:
    ev_gdf = download_ev_stations_nrel(NREL_API_KEY, "TX")
except Exception as e:
    print("EV API error:", e)
    ev_gdf = gpd.GeoDataFrame(columns=["station_name", "geometry"], geometry="geometry", crs=CRS_WGS84)

if len(ev_gdf) > 0:
    minx, miny, maxx, maxy = get_bbox(gdf.to_crs(CRS_WGS84))
    ev_gdf = ev_gdf.cx[minx:maxx, miny:maxy].copy()

ev_gdf_ft = ev_gdf.to_crs(CRS_TEXAS_FT) if len(ev_gdf) > 0 else ev_gdf

def count_points_in_buffers(asset_buffers_gdf, buffer_col, points_gdf, asset_id_col="asset_id"):
    if len(points_gdf) == 0:
        return pd.DataFrame({asset_id_col: asset_buffers_gdf[asset_id_col], "point_count": 0})

    buffers = asset_buffers_gdf[[asset_id_col, buffer_col]].copy()
    buffers = buffers.rename(columns={buffer_col: "geometry"})
    buffers = gpd.GeoDataFrame(buffers, geometry="geometry", crs=points_gdf.crs)

    joined = gpd.sjoin(points_gdf, buffers, predicate="within", how="left")
    counts = joined.groupby(asset_id_col, dropna=True).size().reset_index(name="point_count")

    result = asset_buffers_gdf[[asset_id_col]].merge(counts, on=asset_id_col, how="left")
    result["point_count"] = result["point_count"].fillna(0).astype(int)
    return result

gdf_ft = gdf_ft.drop(columns=[c for c in ["ev_stations_3mi"] if c in gdf_ft.columns], errors="ignore")

ev_counts = count_points_in_buffers(gdf_ft, "buffer_3mi", ev_gdf_ft, "asset_id")
ev_counts = ev_counts.rename(columns={"point_count": "ev_stations_3mi"})

gdf_ft = gdf_ft.merge(ev_counts, on="asset_id", how="left")
gdf_ft["ev_stations_3mi"] = gdf_ft["ev_stations_3mi"].fillna(0).astype(int)

print(f"EV stations in study area bbox: {len(ev_gdf):,}")

EV stations in study area bbox: 121


In [ ]:

# 7. SOLAR + SCORING


for col in [
    "rooftop_generation_kwh_yr", "canopy_generation_kwh_yr", "total_generation_kwh_yr",
    "households_supported_est", "yield_per_household_1mi", "yield_per_household_3mi",
    "score_generation", "score_households", "score_ev_gap", "priority_score"
]:
    if col in gdf_ft.columns:
        gdf_ft = gdf_ft.drop(columns=col)

gdf_ft["rooftop_generation_kwh_yr"] = gdf_ft["Roof_Sq_Ft"].apply(solar_kwh)
gdf_ft["canopy_generation_kwh_yr"] = gdf_ft["Canopy_Usable_Sq_Ft"].apply(solar_kwh)
gdf_ft["total_generation_kwh_yr"] = gdf_ft["rooftop_generation_kwh_yr"] + gdf_ft["canopy_generation_kwh_yr"]

gdf_ft["households_supported_est"] = (gdf_ft["total_generation_kwh_yr"] / AVG_HOUSEHOLD_KWH_PER_YEAR).round(1)

gdf_ft["yield_per_household_1mi"] = np.where(
    gdf_ft["households_1mi"] > 0,
    gdf_ft["total_generation_kwh_yr"] / gdf_ft["households_1mi"],
    np.nan
)
gdf_ft["yield_per_household_3mi"] = np.where(
    gdf_ft["households_3mi"] > 0,
    gdf_ft["total_generation_kwh_yr"] / gdf_ft["households_3mi"],
    np.nan
)

# Priority score explanation:
# 50% energy potential
# 30% households nearby
# 20% EV gap (fewer EV stations nearby => higher opportunity)
gdf_ft["score_generation"] = minmax(gdf_ft["total_generation_kwh_yr"])
gdf_ft["score_households"] = minmax(gdf_ft["households_3mi"])
gdf_ft["score_ev_gap"] = 1 - minmax(gdf_ft["ev_stations_3mi"]) if len(gdf_ft) > 1 else 1

gdf_ft["priority_score"] = (
    0.50 * gdf_ft["score_generation"] +
    0.30 * gdf_ft["score_households"] +
    0.20 * gdf_ft["score_ev_gap"]
).round(4)

final_cols = [
    "asset_id", "Store_Name", "Location", "Asset_Category", "latitude", "longitude",
    "Building_Sq_Ft", "Roof_Sq_Ft", "Lot_Sq_Ft", "Canopy_Usable_Sq_Ft",
    "buffer_1mi_area_sqft", "buffer_1mi_area_sqmi", "buffer_3mi_area_sqft", "buffer_3mi_area_sqmi",
    "households_1mi", "households_3mi", "ev_stations_3mi",
    "rooftop_generation_kwh_yr", "canopy_generation_kwh_yr", "total_generation_kwh_yr",
    "households_supported_est", "yield_per_household_1mi", "yield_per_household_3mi",
    "priority_score"
]

final_df = pd.DataFrame(gdf_ft[final_cols]).copy()
final_df = final_df.sort_values(["priority_score", "total_generation_kwh_yr"], ascending=[False, False]).reset_index(drop=True)
final_df["rank"] = np.arange(1, len(final_df) + 1)


In [ ]:

# 8. KPI CARDS

total_assets = len(final_df)
total_gen = final_df["total_generation_kwh_yr"].sum()
avg_gen = final_df["total_generation_kwh_yr"].mean()
total_households_3mi = final_df["households_3mi"].sum()
total_ev = final_df["ev_stations_3mi"].sum()

html = f"""
<div style="display:flex; gap:15px; flex-wrap:wrap; margin:20px 0;">
  <div style="background:#0f172a; color:white; padding:20px; border-radius:14px; width:220px;">
    <h3>Total Assets</h3><h1>{total_assets:,}</h1>
  </div>
  <div style="background:#14532d; color:white; padding:20px; border-radius:14px; width:220px;">
    <h3>Total Generation</h3><h1>{total_gen:,.0f}</h1><p>kWh/year</p>
  </div>
  <div style="background:#1d4ed8; color:white; padding:20px; border-radius:14px; width:220px;">
    <h3>Avg Generation</h3><h1>{avg_gen:,.0f}</h1><p>kWh/year</p>
  </div>
  <div style="background:#7c2d12; color:white; padding:20px; border-radius:14px; width:220px;">
    <h3>Total Households (3mi)</h3><h1>{total_households_3mi:,}</h1>
  </div>
  <div style="background:#581c87; color:white; padding:20px; border-radius:14px; width:220px;">
    <h3>EV Stations (3mi)</h3><h1>{total_ev:,}</h1>
  </div>
</div>
"""
display(HTML(html))

display(Markdown("""
### Priority Score Formula
**Priority Score = 50% Generation Potential + 30% Households Nearby + 20% EV Gap**

- Higher energy potential increases score
- More households within 3 miles increases score
- Fewer nearby EV stations increases opportunity score
- 1 household support estimate assumes **12,000 kWh/year per household**
"""))


### Priority Score Formula
**Priority Score = 50% Generation Potential + 30% Households Nearby + 20% EV Gap**

- Higher energy potential increases score
- More households within 3 miles increases score
- Fewer nearby EV stations increases opportunity score
- 1 household support estimate assumes **12,000 kWh/year per household**


In [ ]:

# 9. CHARTS


top15 = final_df.head(15).copy()

fig1 = px.bar(
    top15,
    x="Store_Name",
    y="total_generation_kwh_yr",
    color="Asset_Category",
    text=top15["total_generation_kwh_yr"].apply(human_readable_kwh),
    title="Top 15 Assets by Total Solar Generation",
    hover_data=["rank", "households_3mi", "ev_stations_3mi", "priority_score"]
)
fig1.update_layout(xaxis_tickangle=-45, height=550)
fig1.show()

cat_summary = final_df.groupby("Asset_Category", as_index=False).agg({
    "asset_id": "count",
    "total_generation_kwh_yr": "sum",
    "households_3mi": "sum"
}).rename(columns={"asset_id": "asset_count"})

fig2 = px.bar(
    cat_summary,
    x="Asset_Category",
    y="total_generation_kwh_yr",
    text="asset_count",
    color="Asset_Category",
    title="Generation Potential by Asset Category"
)
fig2.update_layout(height=500)
fig2.show()

fig3 = px.scatter(
    final_df,
    x="households_3mi",
    y="total_generation_kwh_yr",
    size="Building_Sq_Ft",
    color="Asset_Category",
    hover_name="Store_Name",
    hover_data=["rank", "priority_score"],
    title="Generation vs Nearby Households (3 miles)"
)
fig3.update_layout(height=600)
fig3.show()

fig4 = px.scatter(
    final_df,
    x="ev_stations_3mi",
    y="priority_score",
    size="total_generation_kwh_yr",
    color="Asset_Category",
    hover_name="Store_Name",
    hover_data=["rank", "households_3mi"],
    title="Priority Score vs EV Stations Nearby"
)
fig4.update_layout(height=600)
fig4.show()

fig5 = px.histogram(
    final_df,
    x="total_generation_kwh_yr",
    nbins=30,
    color="Asset_Category",
    title="Distribution of Total Generation"
)
fig5.update_layout(height=500)
fig5.show()

corr_cols = [
    "Building_Sq_Ft", "Roof_Sq_Ft", "Lot_Sq_Ft", "Canopy_Usable_Sq_Ft",
    "households_1mi", "households_3mi", "ev_stations_3mi",
    "rooftop_generation_kwh_yr", "canopy_generation_kwh_yr",
    "total_generation_kwh_yr", "priority_score"
]
corr = final_df[corr_cols].corr().round(2)

fig6 = ff.create_annotated_heatmap(
    z=corr.values,
    x=list(corr.columns),
    y=list(corr.index),
    annotation_text=corr.values.astype(str),
    showscale=True
)
fig6.update_layout(title="Correlation Heatmap", height=800)
fig6.show()


In [ ]:

# 10. TOP TABLE

display(
    final_df[[
        "rank", "Store_Name", "Asset_Category", "Building_Sq_Ft",
        "households_1mi", "households_3mi", "ev_stations_3mi",
        "total_generation_kwh_yr", "households_supported_est", "priority_score"
    ]].head(25).style.background_gradient(cmap="YlGnBu")
)

,rank,Store_Name,Asset_Category,Building_Sq_Ft,households_1mi,households_3mi,ev_stations_3mi,total_generation_kwh_yr,households_supported_est,priority_score
0,1,"Asset_0_(5,006,529 sqft)",big_box,5006529.121035,3090,47102,18,370338414.167969,30861.500000,0.792500
1,2,"Asset_216_(12,595 sqft)",unknown,12595.419510,6916,49943,12,827532.033683,69.000000,0.377000
2,3,"Asset_152_(15,289 sqft)",unknown,15289.283591,4415,52517,16,1004521.678164,83.700000,0.375100
3,4,"Asset_151_(11,634 sqft)",unknown,11634.305843,4391,52548,16,764385.875921,63.700000,0.375100
4,5,"Asset_40_(58,452 sqft)",other_commercial,58452.317291,6843,50106,13,3840377.445389,320.000000,0.374800
5,6,"Asset_153_(11,954 sqft)",unknown,11954.812724,4483,52502,16,785443.508086,65.500000,0.374600
6,7,Walgreens,unknown,15237.895240,6610,49886,13,1001145.410567,83.400000,0.368300
7,8,The Parks Mall at Arlington,strip_mall,869165.853754,3846,41269,13,77604355.434865,6467.000000,0.367600
8,9,"Asset_41_(52,736 sqft)",other_commercial,52736.677244,6633,49246,13,3464854.007809,288.700000,0.363800
9,10,"Asset_219_(252,540 sqft)",big_box,252540.158212,4106,46626,14,18680670.669249,1556.700000,0.344400


In [ ]:

# 11. INTERACTIVE MAP


assets_wgs = gdf_ft.to_crs(CRS_WGS84).copy()
center_lat = assets_wgs.geometry.y.mean()
center_lon = assets_wgs.geometry.x.mean()

m = folium.Map(location=[center_lat, center_lon], zoom_start=11, tiles="CartoDB positron")

Fullscreen().add_to(m)
MiniMap(toggle_display=True).add_to(m)

marker_cluster = MarkerCluster(name="Asset Locations").add_to(m)

gen_min = assets_wgs["total_generation_kwh_yr"].min()
gen_max = assets_wgs["total_generation_kwh_yr"].max()

def marker_radius(val, vmin, vmax, min_r=4, max_r=15):
    if vmax == vmin:
        return 8
    return min_r + (val - vmin) * (max_r - min_r) / (vmax - vmin)

def marker_color(score):
    if score >= 0.60:
        return "red"
    elif score >= 0.40:
        return "orange"
    elif score >= 0.25:
        return "green"
    else:
        return "blue"

for _, row in assets_wgs.iterrows():
    popup_html = f"""
    <div style="width:300px">
    <h4>{row['Store_Name']}</h4>
    <b>Category:</b> {row['Asset_Category']}<br>
    <b>Location:</b> {row['Location']}<br>
    <b>Building Sq Ft:</b> {row['Building_Sq_Ft']:,.0f}<br>
    <b>Roof Sq Ft:</b> {row['Roof_Sq_Ft']:,.0f}<br>
    <b>Lot Sq Ft (est):</b> {row['Lot_Sq_Ft']:,.0f}<br>
    <b>Households 1 mi:</b> {row['households_1mi']:,}<br>
    <b>Households 3 mi:</b> {row['households_3mi']:,}<br>
    <b>EV Stations 3 mi:</b> {row['ev_stations_3mi']:,}<br>
    <b>Total Generation:</b> {row['total_generation_kwh_yr']:,.0f} kWh/yr<br>
    <b>Households Supported:</b> {row['households_supported_est']:,.1f}<br>
    <b>Priority Score:</b> {row['priority_score']:.3f}
    </div>
    """

    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=marker_radius(row["total_generation_kwh_yr"], gen_min, gen_max),
        popup=folium.Popup(popup_html, max_width=320),
        tooltip=f"{row['Store_Name']} | Score: {row['priority_score']:.3f}",
        color=marker_color(row["priority_score"]),
        fill=True,
        fill_opacity=0.75
    ).add_to(marker_cluster)

top10 = gdf_ft.sort_values("priority_score", ascending=False).head(10).copy().to_crs(CRS_WGS84)

fg1 = folium.FeatureGroup(name="Top 10 - 1 Mile Buffers", show=False)
for _, row in top10.iterrows():
    folium.GeoJson(
        row["buffer_1mi"].__geo_interface__,
        style_function=lambda x: {"color": "green", "weight": 2, "fillOpacity": 0.08},
        tooltip=f"{row['Store_Name']} - 1 mile"
    ).add_to(fg1)
fg1.add_to(m)

fg3 = folium.FeatureGroup(name="Top 10 - 3 Mile Buffers", show=False)
for _, row in top10.iterrows():
    folium.GeoJson(
        row["buffer_3mi"].__geo_interface__,
        style_function=lambda x: {"color": "red", "weight": 2, "fillOpacity": 0.05},
        tooltip=f"{row['Store_Name']} - 3 mile"
    ).add_to(fg3)
fg3.add_to(m)

if len(ev_gdf) > 0:
    ev_layer = folium.FeatureGroup(name="Public EV Stations", show=False)
    for _, row in ev_gdf.iterrows():
        folium.CircleMarker(
            location=[row.geometry.y, row.geometry.x],
            radius=3,
            color="purple",
            fill=True,
            fill_opacity=0.9,
            popup=row.get("station_name", "EV Station")
        ).add_to(ev_layer)
    ev_layer.add_to(m)

# Legend
legend_html = """
<div style="
position: fixed;
bottom: 30px; left: 30px; width: 260px; z-index:9999;
background-color: white;
border:2px solid grey;
border-radius:10px;
padding: 12px;
font-size:14px;
box-shadow: 2px 2px 6px rgba(0,0,0,0.3);
">
<b>Map Legend</b><br><br>
<span style="color:red;">●</span> High Priority Score (>= 0.60)<br>
<span style="color:orange;">●</span> Medium-High Priority (0.40 - 0.59)<br>
<span style="color:green;">●</span> Medium Priority (0.25 - 0.39)<br>
<span style="color:blue;">●</span> Lower Priority (< 0.25)<br>
<span style="color:purple;">●</span> Public EV Stations<br>
<br>
<b>Marker Size:</b> Larger = Higher energy generation<br>
<b>Green Buffer:</b> 1-mile zone<br>
<b>Red Buffer:</b> 3-mile zone
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

folium.LayerControl(collapsed=False).add_to(m)

display(m)


In [ ]:

# 12.  INSIGHTS


top_asset = final_df.iloc[0]
top5_assets = final_df.head(5)[[
    "rank", "Store_Name", "Asset_Category", "total_generation_kwh_yr",
    "households_3mi", "ev_stations_3mi", "priority_score"
]].copy()

cat_mix = final_df["Asset_Category"].value_counts().reset_index()
cat_mix.columns = ["Asset_Category", "Count"]

unknown_pct = (final_df["Asset_Category"].eq("unknown").mean() * 100)

display(Markdown("## Key Insights"))

insights_md = f"""
1. **Highest energy potential asset:** `{top_asset['Store_Name']}` with approximately **{top_asset['total_generation_kwh_yr']:,.0f} kWh/year**.

2. **Top-ranked opportunity site:** `{top_asset['Store_Name']}` also has the highest priority score of **{top_asset['priority_score']:.3f}**.

3. **Household service potential:** The top site could support roughly **{top_asset['households_supported_est']:,.0f} households per year** using the 12,000 kWh/household assumption.

4. **Nearby demand matters:** The strongest positive relationship in the dashboard is between **households within 3 miles** and **priority score**, which means areas with more nearby households tend to rank higher.

5. **EV gap is still useful:** Sites with fewer nearby EV stations can still score well if they have strong generation potential and household reach.

6. **Classification quality:** About **{unknown_pct:.1f}%** of assets are still classified as `unknown`. Reducing this would improve your final report quality.
"""
display(Markdown(insights_md))

display(Markdown("## Top 5 Recommended Assets"))
display(top5_assets.style.background_gradient(cmap="YlGnBu"))

## Key Insights


1. **Highest energy potential asset:** `Asset_0_(5,006,529 sqft)` with approximately **370,338,414 kWh/year**.

2. **Top-ranked opportunity site:** `Asset_0_(5,006,529 sqft)` also has the highest priority score of **0.792**.

3. **Household service potential:** The top site could support roughly **30,862 households per year** using the 12,000 kWh/household assumption.

4. **Nearby demand matters:** The strongest positive relationship in the dashboard is between **households within 3 miles** and **priority score**, which means areas with more nearby households tend to rank higher.

5. **EV gap is still useful:** Sites with fewer nearby EV stations can still score well if they have strong generation potential and household reach.

6. **Classification quality:** About **40.3%** of assets are still classified as `unknown`. Reducing this would improve your final report quality.


## Top 5 Recommended Assets

,rank,Store_Name,Asset_Category,total_generation_kwh_yr,households_3mi,ev_stations_3mi,priority_score
0,1,"Asset_0_(5,006,529 sqft)",big_box,370338414.167969,47102,18,0.792500
1,2,"Asset_216_(12,595 sqft)",unknown,827532.033683,49943,12,0.377000
2,3,"Asset_152_(15,289 sqft)",unknown,1004521.678164,52517,16,0.375100
3,4,"Asset_151_(11,634 sqft)",unknown,764385.875921,52548,16,0.375100
4,5,"Asset_40_(58,452 sqft)",other_commercial,3840377.445389,50106,13,0.374800


In [ ]:

# 13. SAVE OUTPUTS

csv_path = os.path.join(OUTPUT_DIR, "asset_energy_summary.csv")
top25_path = os.path.join(OUTPUT_DIR, "top25_priority_assets.csv")
assets_geo_path = os.path.join(OUTPUT_DIR, "assets_enriched.geojson")
bg_geo_path = os.path.join(OUTPUT_DIR, "household_block_groups.geojson")
html_map_path = os.path.join(OUTPUT_DIR, "asset_energy_map.html")
insights_txt_path = os.path.join(OUTPUT_DIR, "project_insights.txt")

final_df.to_csv(csv_path, index=False)
final_df.head(25).to_csv(top25_path, index=False)
assets_wgs[final_cols + ["geometry"]].to_file(assets_geo_path, driver="GeoJSON")
bg_gdf.to_file(bg_geo_path, driver="GeoJSON")
m.save(html_map_path)

if len(ev_gdf) > 0:
    ev_geo_path = os.path.join(OUTPUT_DIR, "ev_stations.geojson")
    ev_gdf.to_file(ev_geo_path, driver="GeoJSON")

with open(insights_txt_path, "w") as f:
    f.write("PROJECT INSIGHTS\n")
    f.write("=" * 60 + "\n")
    f.write(insights_md.replace("**", "").replace("`", "") + "\n\n")
    f.write("TOP 5 RECOMMENDED ASSETS\n")
    f.write(top5_assets.to_string(index=False))

with open(os.path.join(OUTPUT_DIR, "run_metadata.json"), "w") as f:
    json.dump({
        "input_file": BUILDINGS_CSV,
        "total_assets": int(len(final_df)),
        "output_csv": csv_path,
        "top25_csv": top25_path,
        "assets_geojson": assets_geo_path,
        "household_bg_geojson": bg_geo_path,
        "interactive_map": html_map_path,
        "insights_file": insights_txt_path
    }, f, indent=2)

print("\n============================================================")
print("UPDATED PIPELINE COMPLETED SUCCESSFULLY")
print("============================================================")
print(f"Final CSV: {csv_path}")
print(f"Top 25 CSV: {top25_path}")
print(f"Assets GeoJSON: {assets_geo_path}")
print(f"Block Groups GeoJSON: {bg_geo_path}")
print(f"Interactive Map HTML: {html_map_path}")
print(f"Insights TXT: {insights_txt_path}")

print("\nTop 10 priority assets:")
print(final_df[[
    "rank", "Store_Name", "Asset_Category", "Building_Sq_Ft",
    "households_1mi", "households_3mi", "ev_stations_3mi",
    "total_generation_kwh_yr", "priority_score"
]].head(10).to_string(index=False))


UPDATED PIPELINE COMPLETED SUCCESSFULLY
Final CSV: /content/arlington_energy_outputs/asset_energy_summary.csv
Top 25 CSV: /content/arlington_energy_outputs/top25_priority_assets.csv
Assets GeoJSON: /content/arlington_energy_outputs/assets_enriched.geojson
Block Groups GeoJSON: /content/arlington_energy_outputs/household_block_groups.geojson
Interactive Map HTML: /content/arlington_energy_outputs/asset_energy_map.html
Insights TXT: /content/arlington_energy_outputs/project_insights.txt

Top 10 priority assets:
 rank                  Store_Name   Asset_Category  Building_Sq_Ft  households_1mi  households_3mi  ev_stations_3mi  total_generation_kwh_yr  priority_score
    1    Asset_0_(5,006,529 sqft)          big_box    5.006529e+06            3090           47102               18             3.703384e+08          0.7925
    2     Asset_216_(12,595 sqft)          unknown    1.259542e+04            6916           49943               12             8.275320e+05          0.3770
    3     Ass